# ReaderAdapt-Liver on LiTS17 with the ViT-UNETR 2.5D (Colab, GPU)

Runtime -> Change runtime type -> **GPU**. Same Kaggle token setup as the ViT-UNETR notebook
(Secrets: `KAGGLE_API_TOKEN`, notebook access on).

Pipeline: download LiTS -> leakage-safe split -> train the 3-class ViT-UNETR (optionally warm-started from your
2-class liver checkpoint) -> agentic first read on 51 held-out cases -> calibration -> reader experiment -> figures.
At the end, download the run folder and open it with the Streamlit app on your laptop.

In [ ]:
# 1. Get the code (replace with your GitHub URL once pushed; or upload the zip and unzip it)
REPO_URL = "https://github.com/<your-user>/readeradapt-liver.git"
import os
if not os.path.exists("readeradapt-liver"):
    if os.path.exists("readeradapt-liver.zip"):
        !unzip -q readeradapt-liver.zip
    else:
        !git clone -q $REPO_URL
%cd readeradapt-liver
!pip install -q -r requirements.txt -r requirements-deep.txt kagglehub

In [ ]:
import torch
print(torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 2. LiTS from Kaggle (two datasets: part 1 = masks + volumes pt1-5, part 2 = remaining volumes)
import kagglehub
part1 = kagglehub.dataset_download("andrewmvd/liver-tumor-segmentation")
part2 = kagglehub.dataset_download("andrewmvd/liver-tumor-segmentation-part-2")
print(part1, part2, sep="\n")

### Optional warm start from your 2-class liver model
In the ViT-UNETR notebook, after training, run
```python
torch.save(model.state_dict(), "vitunetr_liver_2class.pt")
```
and upload the file here (or copy it from Drive) as `checkpoints/vitunetr_liver_2class.pt`. Every tensor except the
2-class head is reused; the head is re-initialised for 3 classes. The split below reproduces that notebook's split,
so the checkpoint has never seen a calibration or test case.

In [ ]:
# 3. Write the run config with the real data paths
import yaml, os
cfg = yaml.safe_load(open("configs/lits_vitunetr.yaml"))
cfg["data"]["lits_dirs"] = [part1, part2]
warm = "checkpoints/vitunetr_liver_2class.pt"
cfg["segmenter"]["init_from"] = warm if os.path.exists(warm) else None
yaml.safe_dump(cfg, open("configs/lits_colab.yaml", "w"), sort_keys=False)
print("warm start:", cfg["segmenter"]["init_from"])

In [ ]:
!python -m radagent data --config configs/lits_colab.yaml

In [ ]:
# 4. Train the 3-class ViT-UNETR (~40-60 min on an A100 for 30 epochs; fewer epochs for a first look)
!python -m radagent train --config configs/lits_colab.yaml

In [ ]:
# 5. Agent perception tools on the 51 held-out cases (segmentation with flip-TTA x MC-dropout, measurement)
!python -m radagent infer --config configs/lits_colab.yaml

In [ ]:
!python -m radagent calibrate --config configs/lits_colab.yaml
!python -m radagent first-read --config configs/lits_colab.yaml

In [ ]:
!python -m radagent simulate --config configs/lits_colab.yaml
!python -m radagent figures --config configs/lits_colab.yaml

In [ ]:
# 6. Look at one agent trace
!python -m radagent agent --config configs/lits_colab.yaml

In [ ]:
# 7. Package the run (without the training-slice cache) and download it
!cd outputs && zip -qr ../lits_vitunetr_run.zip lits_vitunetr -x "lits_vitunetr/cache/train_slices/*"
from google.colab import files
files.download("lits_vitunetr_run.zip")

On your laptop: unzip into `readeradapt-liver/outputs/` and run
`streamlit run app/streamlit_app.py -- --run outputs/lits_vitunetr`, then set `RUN` in
`notebooks/evaluation.ipynb` to that folder to regenerate the evaluation.